# onchain-sybil-detector analysis (offline)

This notebook runs fully offline using synthetic data only.

1. Generate synthetic network
2. Extract features and compare legit vs sybil distributions
3. Run HDBSCAN and visualize UMAP projection
4. Compute coordination scores and benchmark metrics
5. Render network graph + HTML report
6. Show explainability evidence


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import umap

from datasets.synthetic_generator import generate_synthetic_sybil_network
from sybil_detector import SybilDetector, extract_features, run_benchmark
from sybil_detector.visualization import generate_report, plot_cluster_graph

tx, labels = generate_synthetic_sybil_network(seed=42)
tx.shape, labels.shape


((26858, 10), (700, 4))

In [2]:
features = extract_features(tx)
joined = features.merge(labels[["address", "is_sybil"]], on="address", how="left")
joined["is_sybil"] = joined["is_sybil"].fillna(0).astype(int)
joined["label"] = joined["is_sybil"].map({0: "legit", 1: "sybil"})

fig = px.histogram(
    joined,
    x="burst_ratio",
    color="label",
    barmode="overlay",
    title="Burst Ratio: Legit vs Sybil",
)
fig.show()

fig2 = px.histogram(
    joined,
    x="gas_price_cv",
    color="label",
    barmode="overlay",
    title="Gas Price CV: Legit vs Sybil",
)
fig2.show()


In [3]:
detector = SybilDetector(min_cluster_size=3, min_samples=2)
clusters = detector.fit_predict(features)

numeric = features.drop(columns=["address", "common_funder_address", "hour_histogram"])
embedding = umap.UMAP(random_state=42).fit_transform(numeric)

proj = pd.DataFrame({"x": embedding[:, 0], "y": embedding[:, 1], "address": features["address"]})
proj = proj.merge(clusters[["address", "cluster_id", "sybil_probability"]], on="address", how="left")
fig = px.scatter(
    proj,
    x="x",
    y="y",
    color=proj["cluster_id"].astype(str),
    size="sybil_probability",
    title="HDBSCAN Clusters on 2D UMAP Projection",
)
fig.show()


/home/fkalghan/.local/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [4]:
metrics = run_benchmark(detector, tx, labels)
metrics


{'address': {'precision': 1.0, 'recall': 0.83, 'f1': 0.907103825136612},
 'cluster': {'precision': 1.0, 'recall': 0.9, 'f1': 0.9473684210526316},
 'baselines': {'same_funder': {'precision': 0.0999000999000999,
   'recall': 0.5,
   'f1': 0.16652789342214822},
  'same_gas': {'precision': 0.7801047120418848,
   'recall': 0.745,
   'f1': 0.7621483375959079},
  'same_timing': {'precision': 0.6548223350253807,
   'recall': 0.645,
   'f1': 0.6498740554156172}},
 'decision_threshold': 0.7,
 'n_addresses': 2321,
 'n_predicted_sybil_addresses': 166,
 'n_predicted_clusters': 10}

In [5]:
graph_path = plot_cluster_graph(tx, clusters, output_html="notebooks/cluster_graph.html")
report_html = generate_report(clusters, format="html")
Path("notebooks/cluster_report.html").write_text(report_html, encoding="utf-8")
graph_path, "notebooks/cluster_report.html"


('notebooks/cluster_graph.html', 'notebooks/cluster_report.html')

In [6]:
cluster_scores = clusters[clusters["cluster_id"] != -1].groupby("cluster_id")["sybil_probability"].mean()
top_cluster = int(cluster_scores.sort_values(ascending=False).index[0])
explanation = detector.explain_cluster(top_cluster)
print(explanation)

example_sentence = (
    "Cluster 3 flagged because: 85% share funder 0xABC, gas CV=0.02, temporal cosine=0.91"
)
print(example_sentence)


Cluster 7 flagged because temporal cosine=0.71, gas similarity=0.98, common funding=1.00, sequential activity=0.79, coordination_score=0.87, size=19.
Cluster 3 flagged because: 85% share funder 0xABC, gas CV=0.02, temporal cosine=0.91
